## Skill- Memory Context

In [ ]:
from urllib.request import urlopen

from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
backend = StateBackend()

from pathlib import Path

skills_content= Path("skills/langgraph/SKILL.md").read_text(encoding="utf-8")

skills_files = {
    "/skills/langgraph/SKILL.md": create_file_data(skills_content),
}

In [4]:
# Read every skills/<name>/SKILL.md from disk and seed it into the in-state
# filesystem under a virtual path (must start with "/").
skill_dirs = ["langgraph", "python", "aws", "report-writer"]
skills_files = {
    f"/skills/{name}/SKILL.md": create_file_data(
        Path(f"skills/{name}/SKILL.md").read_text(encoding="utf-8")
    )
    for name in skill_dirs
}

In [6]:
agent=create_deep_agent(
    model="openai:gpt-5.4",
    backend=backend,
    skills=["/skills/"],
    checkpointer=checkpointer
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "What skills do you have available, and when would you use each?"}],
        
        "files": skills_files,
    },
    config={"configurable": {"thread_id": "skills-demo1"}},
)

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'Available skills:\n\n- aws\n  - Use for AWS architecture, service selection, boto3/AWS CLI, IAM, security, cost optimization, and operational guidance across services like EC2, S3, Lambda, DynamoDB, RDS, ECS/EKS, CloudWatch, Bedrock, and SageMaker.\n\n- langgraph\n  - Use for building or debugging LangGraph workflows: StateGraph design, nodes/edges, routing, persistence, memory, checkpointers, subgraphs, human-in-the-loop, and streaming agent systems.\n\n- python\n  - Use for writing, debugging, refactoring, reviewing, or explaining Python code, including typing, async, packaging, testing, OOP, and general Python best practices.\n\n- report-writer\n  - Use after any substantive answer to produce a structured markdown report of the interaction and save it to a file. Especially relevant when the user wants a report, summary document, or saved output.\n\nHow I decide:\n- I first match your request to a domain.\n- If a skill applies, I read that skill’s `/skills/

In [8]:
print(result["messages"][-1].content[0]['text'])

Available skills:

- aws
  - Use for AWS architecture, service selection, boto3/AWS CLI, IAM, security, cost optimization, and operational guidance across services like EC2, S3, Lambda, DynamoDB, RDS, ECS/EKS, CloudWatch, Bedrock, and SageMaker.

- langgraph
  - Use for building or debugging LangGraph workflows: StateGraph design, nodes/edges, routing, persistence, memory, checkpointers, subgraphs, human-in-the-loop, and streaming agent systems.

- python
  - Use for writing, debugging, refactoring, reviewing, or explaining Python code, including typing, async, packaging, testing, OOP, and general Python best practices.

- report-writer
  - Use after any substantive answer to produce a structured markdown report of the interaction and save it to a file. Especially relevant when the user wants a report, summary document, or saved output.

How I decide:
- I first match your request to a domain.
- If a skill applies, I read that skill’s `/skills/.../SKILL.md`.
- Then I follow its workflow

In [9]:
result = agent.invoke(
      {
          "messages": [{
              "role": "user",
              "content": "How do I build a LangGraph graph with conditional routing and memory? Show a minimal example.",       
          }],
          "files": skills_files,   # ✅  seed skills into THIS thread's state
      },
      config={"configurable": {"thread_id": "skills-demo-2"}},
  )

# Print the conversation so you can SEE the skill being triggered:
# look for a ToolMessage from `read_file` on the langgraph-docs SKILL.md.
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

How do I build a LangGraph graph with conditional routing and memory? Show a minimal example.
================================== Ai Message ==================================

[{'arguments': '{"file_path":"/skills/langgraph/SKILL.md","offset":0,"limit":1000}', 'call_id': 'call_y02qVXPErud42RyiBuVAXXHV', 'name': 'read_file', 'type': 'function_call', 'id': 'fc_0d2a5888ee28951b006a5d1379c5308193ab835ed5fd87dfc1', 'status': 'completed'}, {'arguments': '{"file_path":"/skills/report-writer/SKILL.md","offset":0,"limit":1000}', 'call_id': 'call_B5EnSJAiEZGUUNhtgNoVUZIf', 'name': 'read_file', 'type': 'function_call', 'id': 'fc_0d2a5888ee28951b006a5d1379c54481938ead7c76c553a8f6', 'status': 'completed'}]
Tool Calls:
  read_file (call_y02qVXPErud42RyiBuVAXXHV)
 Call ID: call_y02qVXPErud42RyiBuVAXXHV
  Args:
    file_path: /skills/langgraph/SKILL.md
    offset: 0
    limit: 1000
  read_file (call_B5EnSJAiEZGUUNhtgNoVU

In [10]:
result = agent.invoke(
      {
          "messages": [{
              "role": "user",
              "content": "How do i set up an EC2  insstance in AWS.",       
          }],
          "files": skills_files,   # ✅  seed skills into THIS thread's state
      },
      config={"configurable": {"thread_id": "skills-demo-3"}},
  )

# Print the conversation so you can SEE the skill being triggered:
# look for a ToolMessage from `read_file` on the langgraph-docs SKILL.md.
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

How do i set up an EC2  insstance in AWS.
================================== Ai Message ==================================

[{'arguments': '{"file_path":"/skills/aws/SKILL.md","offset":0,"limit":1000}', 'call_id': 'call_EhodulYYfl13tLBS5qSbAXsM', 'name': 'read_file', 'type': 'function_call', 'id': 'fc_0e3a412b2a1b0607006a5d1502cc2c8193a145eea43b05368d', 'status': 'completed'}, {'arguments': '{"file_path":"/skills/report-writer/SKILL.md","offset":0,"limit":1000}', 'call_id': 'call_m0ThsE8xCm07SS5cJrz00Jvo', 'name': 'read_file', 'type': 'function_call', 'id': 'fc_0e3a412b2a1b0607006a5d1502cc388193826e48867b65a223', 'status': 'completed'}]
Tool Calls:
  read_file (call_EhodulYYfl13tLBS5qSbAXsM)
 Call ID: call_EhodulYYfl13tLBS5qSbAXsM
  Args:
    file_path: /skills/aws/SKILL.md
    offset: 0
    limit: 1000
  read_file (call_m0ThsE8xCm07SS5cJrz00Jvo)
 Call ID: call_m0ThsE8xCm07SS5cJrz00Jvo
  Args:
    file_pa